# 02 Procesamiento de texto, recuperación secuencial e indexación

## Objetivo

Implementar y comparar dos mecanismos de recuperación de información —búsqueda secuencial e índice invertido— sobre colecciones de libros del Proyecto Gutenberg, investigando experimentalmente su rendimiento y escalabilidad.

### Parte A. Corpus pequeño

**1. Preparación del corpus**

Utilice los 18 documentos de la colección Gutenberg disponible en NLTK.

1. Cargue los textos originales.
2. Identifique el número de documentos y sus tamaños.
3. Implemente tokenización y normalización: conversión a minúsculas, tratamiento de puntuación y caracteres especiales.

Seleccione y documente una configuración de preprocesamiento que se utilizará consistentemente en todos los experimentos.

**Carga del dataset e identificación de documentos**

In [16]:
from pathlib import Path
import nltk
from nltk.corpus import gutenberg
import os

ruta = str(Path("../data/nltk_data").resolve())
if ruta not in nltk.data.path:
    nltk.data.path.append(ruta)

documentos = gutenberg.fileids()
print("Cantidad:", len(documentos))

for doc in gutenberg.fileids():
    kb = os.path.getsize(gutenberg.abspath(doc)) / 1024
    print(f"{doc}: {kb:.1f} KB")

Cantidad: 18
austen-emma.txt: 866.3 KB
austen-persuasion.txt: 455.4 KB
austen-sense.txt: 657.2 KB
bible-kjv.txt: 4231.0 KB
blake-poems.txt: 37.3 KB
bryant-stories.txt: 243.6 KB
burgess-busterbrown.txt: 82.7 KB
carroll-alice.txt: 141.0 KB
chesterton-ball.txt: 446.7 KB
chesterton-brown.txt: 397.1 KB
chesterton-thursday.txt: 313.0 KB
edgeworth-parents.txt: 913.2 KB
melville-moby_dick.txt: 1213.9 KB
milton-paradise.txt: 457.2 KB
shakespeare-caesar.txt: 109.7 KB
shakespeare-hamlet.txt: 159.1 KB
shakespeare-macbeth.txt: 98.0 KB
whitman-leaves.txt: 694.5 KB


**Tokenización y limpieza de caracteres**

Configuración aplicada en todos los experimentos:
- Se convierten los textos a minúsculas antes de tokenizar.
- `RegexpTokenizer(r"[A-Za-z]+(?:['’][A-Za-z]+)?")` conserva secuencias de letras ASCII y apóstrofos internos; otros signos, incluidos los guiones, separan tokens.
- No se eliminan stopwords ni se aplica stemming o lematización.
- El patrón no incluye dígitos ni letras acentuadas; esta limitación se mantiene igual para los documentos y las consultas.

In [ ]:
from nltk.tokenize import RegexpTokenizer

tokenizador = RegexpTokenizer(r"[A-Za-z]+(?:['’][A-Za-z]+)?")
corpus_procesado = {}

for doc_id in gutenberg.fileids():
    texto = gutenberg.raw(doc_id)
    corpus_procesado[doc_id] = tokenizador.tokenize(texto.lower())

**2. Búsqueda secuencial**

Implemente una función que reciba un término de consulta y devuelva los identificadores de todos los documentos que lo contienen con una búsqueda secuencial.

La implementación deberá utilizar obligatoriamente dos lazos `for` anidados: el primero recorre los documentos y el segundo los tokens de cada documento.

No es necesario utilizar índices invertidos, motores de búsqueda ni estructuras auxiliares que almacenen anticipadamente los documentos asociados a cada término.

Ejecute consultas con términos frecuentes, poco frecuentes e inexistentes.

In [ ]:
def busqueda_secuencial(termino):
    termino = termino.lower()
    resultado = []

    for doc_id, tokens in corpus_procesado.items():
        for token in tokens:
            if token == termino:
                resultado.append(doc_id)
                break
    return resultado

In [ ]:
consultas = {
    "frecuente": "he",
    "poco frecuente": "moby",
    "inexistente": "tinitana",
}

for categoria, termino in consultas.items():
    resultados = busqueda_secuencial(termino)
    print(f"{categoria} ({termino}): DF={len(resultados)}; documentos={resultados}")

**3. Medición de tiempos de ejecución**

Utilice `time.perf_counter_ns()` o una herramienta de medición equivalente para obtener el tiempo de ejecución de las consultas.

Cada medición deberá corresponder a **30 ejecuciones de la misma consulta**. Y se debe calcular la media aritmética.

Registre los resultados en milisegundos. No incluya la carga de archivos ni la tokenización dentro del tiempo de consulta.

In [ ]:
import time

def medir_consulta(funcion_busqueda, termino, repeticiones=30):
    tiempos_ms = []

    for _ in range(repeticiones):
        inicio = time.perf_counter_ns()
        funcion_busqueda(termino)
        fin = time.perf_counter_ns()

        tiempo_ms = (fin - inicio) / 1_000_000
        tiempos_ms.append(tiempo_ms)

    return sum(tiempos_ms) / repeticiones

In [ ]:
for categoria, termino in consultas.items():
    promedio_ms = medir_consulta(busqueda_secuencial, termino, repeticiones=30)
    cantidad_documentos = len(busqueda_secuencial(termino))
    print(
        f"{categoria} ({termino}): {cantidad_documentos} documentos; "
        f"promedio = {promedio_ms:.4f} ms"
    )

**4. Construcción del índice invertido**

Implemente un índice invertido que contenga el vocabulario de la colección y las listas de documentos asociados a cada término.

Inspeccione las listas de postings y calcule la frecuencia documental (DF) de varios términos.

Implemente consultas de un término y consultas booleanas con AND, OR y NOT.

In [37]:
indice_invertido = {}

for doc_id, tokens in corpus_procesado.items():
    for termino in set(tokens):
        if termino not in indice_invertido:
            indice_invertido[termino] = []
        
        indice_invertido[termino].append(doc_id)

In [46]:
import pandas as pd

terminos = sorted(indice_invertido.keys())
documentos = list(corpus_procesado.keys())

matriz = pd.DataFrame(
    0,
    index=terminos,
    columns=documentos,
    dtype="int8"
)

for termino, postings in indice_invertido.items():
    for doc_id in postings:
        matriz.at[termino, doc_id] = 1

print("Tamaño completo:", matriz.shape)
display(matriz.iloc[:18, :18])

Tamaño completo: (43495, 18)


,austen-emma.txt,austen-persuasion.txt,austen-sense.txt,bible-kjv.txt,blake-poems.txt,bryant-stories.txt,burgess-busterbrown.txt,carroll-alice.txt,chesterton-ball.txt,chesterton-brown.txt,chesterton-thursday.txt,edgeworth-parents.txt,melville-moby_dick.txt,milton-paradise.txt,shakespeare-caesar.txt,shakespeare-hamlet.txt,shakespeare-macbeth.txt,whitman-leaves.txt
a,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
a'lee,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
a'low,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
a'm,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
a'most,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0
a'n,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
a'ready,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
a'shiver,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
a'th,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
a'top,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0


In [ ]:
def consulta_termino(termino):
    return indice_invertido.get(termino.lower(), []).copy()

def consulta_and(termino_1, termino_2):
    postings_1 = set(consulta_termino(termino_1))
    postings_2 = set(consulta_termino(termino_2))
    return sorted(postings_1 & postings_2)

def consulta_or(termino_1, termino_2):
    postings_1 = set(consulta_termino(termino_1))
    postings_2 = set(consulta_termino(termino_2))
    return sorted(postings_1 | postings_2)

def consulta_not(termino):
    universo = set(documentos)
    postings = set(consulta_termino(termino))
    return sorted(universo - postings)

terminos_df = ["he", "science", "moby", "tinitana"]
resumen_postings = []
for termino in terminos_df:
    postings = consulta_termino(termino)
    resumen_postings.append({"término": termino, "postings": postings, "DF": len(postings)})
display(pd.DataFrame(resumen_postings))

print("Consulta de un término (moby):", consulta_termino("moby"))
print("science AND whale:", consulta_and("science", "whale"))
print("science OR whale:", consulta_or("science", "whale"))
print("NOT science:", consulta_not("science"))

**5. Comparación**

Ejecute las mismas consultas con ambos métodos.

Compruebe que recuperan exactamente los mismos identificadores de documentos y mida el tiempo medio de 30 ejecuciones para cada consulta y método.

La carga del corpus, la tokenización y la construcción del índice ocurren antes de medir las consultas.

In [ ]:
for categoria, termino in consultas.items():
    resultados_secuencial = busqueda_secuencial(termino)
    resultados_indice = consulta_termino(termino)
    mismos_documentos = resultados_secuencial == resultados_indice

    tiempo_secuencial_ms = medir_consulta(
        busqueda_secuencial, termino, repeticiones=30
    )
    tiempo_indice_ms = medir_consulta(
        consulta_termino, termino, repeticiones=30
    )

    print(
        f"{categoria} ({termino}): mismos documentos={mismos_documentos}; "
        f"cantidad={len(resultados_secuencial)}; "
        f"secuencial={tiempo_secuencial_ms:.6f} ms; "
        f"índice={tiempo_indice_ms:.6f} ms"
    )

### Parte B. Corpus grande

**6. Descarga de Gutenberg**

Descargue el archivo oficial `txt-files.tar.zip`, de aproximadamente 11 GB, desde:

https://www.gutenberg.org/cache/epub/feeds/

Identifique los archivos TXT, evite contar distintas versiones del mismo libro como documentos independientes y registre la cantidad de documentos válidos.

**7. Recuperación sobre el corpus grande**

Reutilice las funciones desarrolladas en la Parte A.

1. Procese los documentos con la misma configuración de normalización.
2. Ejecute las consultas mediante búsqueda secuencial y mida los tiempos.
3. Construya el índice invertido.
4. Repita las consultas utilizando el índice.
5. Compruebe la igualdad de resultados y compare los tiempos.

Trabaje con subconjuntos de la colección de tamaños crecientes, conservando los identificadores originales. Documente los recursos de memoria y almacenamiento utilizados.

### Parte C. Análisis experimental de escalabilidad

**8. Tiempo de ejecución en función del corpus**

Seleccione al menos cinco tamaños crecientes del corpus grande. Como referencia, pueden utilizarse 50, 100, 250, 500 y 1.000 documentos, ampliándolos si el equipo lo permite.

Para cada tamaño $n$:

1. Ejecute una misma consulta mediante búsqueda secuencial.
2. Repítala 30 veces y calcule el tiempo promedio.
3. Construya el índice invertido correspondiente a ese subconjunto.
4. Ejecute la consulta mediante el índice, también 30 veces.
5. Registre ambos tiempos y el número total de tokens.

Elabore una **gráfica del tiempo medio de ejecución en función del número de documentos**, colocando el número de documentos en el eje X y el tiempo en milisegundos en el eje Y. Represente los dos métodos en la misma gráfica.

**9. Estimación de una función de tiempo**

Utilizando los resultados experimentales, determine una función:

$t=f(n)$

para cada método de recuperación.

Como primera aproximación, utilice un ajuste lineal por mínimos cuadrados:

$t(n)=an+b$

Estime los parámetros $a$ y $b$, reporte el coeficiente de determinación $R^2$ y represente las funciones ajustadas junto a los datos experimentales.

Explique si el modelo lineal representa adecuadamente las mediciones y cómo se relaciona con la complejidad computacional de cada algoritmo.